In [2]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 37.9 MB/s  0:00:02


In [17]:
import pandas as pd
import numpy as np

from catboost import CatBoostClassifier

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score

Load files

In [5]:
train = pd.read_csv("train (2).csv")
test = pd.read_csv("test (2).csv")
sample_submission = pd.read_csv("sample_submission.csv")

print("Train:", train.shape)
print("Test:", test.shape)
print("Sample submission:", sample_submission.shape)

train.head()

Train: (699960, 26)
Test: (299984, 25)
Sample submission: (299984, 2)


,id,amount_usd,hours_since_last_txn,txn_count_last_24h,distance_from_home_km,card_age_months,customer_age,account_balance_usd,cvv_retry_count,velocity_score,...,auth_method,channel,device_type,is_foreign_transaction,is_new_merchant,used_vpn,ip_country_mismatch,billing_shipping_mismatch,is_ai_generated_scam_attempt,is_fraud
0,0,61.16,9.77,1,17.78,48,56,1269.35,0,5.5,...,OTP,POS,iPhone,False,False,False,False,False,False,0.0
1,1,24.83,1.54,3,13.06,38,28,3233.00,0,33.4,...,PIN,In-App,Android Phone,False,False,False,False,False,False,0.0
2,2,6.36,1.20,4,8.17,50,55,1046.88,0,35.8,...,OTP,Online,Tablet,False,False,True,False,False,False,0.0
3,3,37.44,5.13,4,8.67,48,48,2217.67,0,20.7,...,3D Secure,POS,Android Phone,False,False,False,False,False,False,0.0
4,4,653.85,0.80,4,11.69,37,52,3291.06,0,34.8,...,3D Secure,Online,Windows PC,False,False,False,False,False,False,0.0


Check target, IDs, metric and allowed data

In [6]:
target = "is_fraud"
id_col = "id"

print("Target:", target)
print("Metric: ROC-AUC")
print("ID column:", id_col)

print("Train columns:", train.columns.tolist())
print("Test columns:", test.columns.tolist())

print("Target distribution:")
print(train[target].value_counts(normalize=True))

Target: is_fraud
Metric: ROC-AUC
ID column: id
Train columns: ['id', 'amount_usd', 'hours_since_last_txn', 'txn_count_last_24h', 'distance_from_home_km', 'card_age_months', 'customer_age', 'account_balance_usd', 'cvv_retry_count', 'velocity_score', 'time_of_day_hour', 'day_of_week', 'merchant_risk_score', 'prior_disputes', 'merchant_category', 'card_type', 'auth_method', 'channel', 'device_type', 'is_foreign_transaction', 'is_new_merchant', 'used_vpn', 'ip_country_mismatch', 'billing_shipping_mismatch', 'is_ai_generated_scam_attempt', 'is_fraud']
Test columns: ['id', 'amount_usd', 'hours_since_last_txn', 'txn_count_last_24h', 'distance_from_home_km', 'card_age_months', 'customer_age', 'account_balance_usd', 'cvv_retry_count', 'velocity_score', 'time_of_day_hour', 'day_of_week', 'merchant_risk_score', 'prior_disputes', 'merchant_category', 'card_type', 'auth_method', 'channel', 'device_type', 'is_foreign_transaction', 'is_new_merchant', 'used_vpn', 'ip_country_mismatch', 'billing_shippi

Run OOF prediction with default CatBoost

In [7]:
X = train.drop(columns=[target, id_col])
y = train[target]

X_test = test.drop(columns=[id_col])

cat_features = X.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

for col in cat_features:
    X[col] = X[col].fillna("Missing").astype(str)
    X_test[col] = X_test[col].fillna("Missing").astype(str)

print("Categorical features:", cat_features)

Categorical features: ['merchant_category', 'card_type', 'auth_method', 'channel', 'device_type']


Define OOF function

In [8]:
def oof_predict(model, X, y, cat_features, n_splits=5):

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=42
    )

    oof_preds = np.zeros(len(X))
    fold_scores = []

    for fold, (train_idx, valid_idx) in enumerate(
        skf.split(X, y), 1
    ):
        X_train = X.iloc[train_idx]
        X_valid = X.iloc[valid_idx]

        y_train = y.iloc[train_idx]
        y_valid = y.iloc[valid_idx]

        model.fit(
            X_train,
            y_train,
            cat_features=cat_features,
            verbose=False
        )

        preds = model.predict_proba(X_valid)[:, 1]

        oof_preds[valid_idx] = preds

        score = roc_auc_score(y_valid, preds)
        fold_scores.append(score)

        print(f"Fold {fold}: ROC-AUC = {score:.6f}")

    overall_score = roc_auc_score(y, oof_preds)

    print("\nOverall OOF ROC-AUC:", overall_score)
    print(
        "Fold mean ± std:",
        f"{np.mean(fold_scores):.6f} ± {np.std(fold_scores):.6f}"
    )

    return oof_preds, fold_scores, overall_score

Run default CatBoost OOF

In [9]:
model = CatBoostClassifier(
    iterations=1500,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=False
)

oof_preds, fold_scores, oof_score = oof_predict(
    model,
    X,
    y,
    cat_features,
    n_splits=5
)

Fold 1: ROC-AUC = 0.944425
Fold 2: ROC-AUC = 0.947394
Fold 3: ROC-AUC = 0.950738
Fold 4: ROC-AUC = 0.946150
Fold 5: ROC-AUC = 0.941625

Overall OOF ROC-AUC: 0.9460220229245
Fold mean ± std: 0.946067 ± 0.003034


Adversarial validation

Prepare adversarial dataset

In [10]:
adv_data = pd.concat(
    [X, X_test],
    axis=0,
    ignore_index=True
)

adv_target = np.concatenate([
    np.zeros(len(X)),
    np.ones(len(X_test))
])

print(adv_data.shape)

(999944, 24)


Run adversarial validation

In [11]:
X_adv_train, X_adv_valid, y_adv_train, y_adv_valid = train_test_split(
    adv_data,
    adv_target,
    test_size=0.20,
    stratify=adv_target,
    random_state=42
)

adv_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=False
)

adv_model.fit(
    X_adv_train,
    y_adv_train,
    cat_features=cat_features
)

adv_preds = adv_model.predict_proba(X_adv_valid)[:, 1]

adv_auc = roc_auc_score(y_adv_valid, adv_preds)

print("Adversarial Validation AUC:", adv_auc)

Adversarial Validation AUC: 0.49837737019906214


Build submission.csv

Train final model

In [12]:
final_model = CatBoostClassifier(
    iterations=1500,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=False
)

final_model.fit(
    X,
    y,
    cat_features=cat_features
)

test_preds = final_model.predict_proba(X_test)[:, 1]

Create submission

In [13]:
submission = pd.DataFrame({
    "id": test["id"],
    "is_fraud": test_preds
})

submission.to_csv("submission.csv", index=False)

submission.head()

,id,is_fraud
0,699960,0.000148
1,699961,0.000063
2,699962,0.072149
3,699963,0.000880
4,699964,0.001366


Check submission

In [14]:
def check_submission(submission, sample_submission):

    assert submission.shape == sample_submission.shape, \
        "Shape mismatch"

    assert submission.columns.tolist() == sample_submission.columns.tolist(), \
        "Column mismatch"

    assert submission["id"].equals(sample_submission["id"]), \
        "ID mismatch or incorrect ordering"

    assert submission.isnull().sum().sum() == 0, \
        "Submission contains missing values"

    assert submission["is_fraud"].between(0, 1).all(), \
        "Predictions must be between 0 and 1"

    print("Submission checks passed!")
    print("Shape:", submission.shape)
    print("Columns:", submission.columns.tolist())

In [15]:
check_submission(submission, sample_submission)

Submission checks passed!
Shape: (299984, 2)
Columns: ['id', 'is_fraud']


In [18]:
!kaggle competitions submit -c playground-series-s6e11 -f submission.csv -m "Default CatBoost OOF baseline"

You must authenticate before you can call the Kaggle API.
Follow the instructions to authenticate at: https://github.com/Kaggle/kaggle-cli/blob/main/docs/README.md#authentication
